# Kisan Dost — apna AI model train karein (Google Colab, free)

Yeh notebook aapki **label ki hui photos** se bimari pehchanne wala model banata hai.
Model **phone ke andar** chalta hai (TensorFlow.js) — kisi API ki zaroorat nahi.

**Qadam:**
1. Server par `npm run dataset` → `dataset/` folder banega → use **zip** karein (`dataset.zip`)
2. Colab mein: Runtime → Change runtime type → **GPU**
3. Neeche ke cells ek ek kar ke chalayein
4. Aakhir mein `kisan_model.zip` download hoga → unzip karke files app ke **`public/model/`** mein daalein → `npm run build`

**Folder ka naam = label.** Naam app ki bimari IDs jaise hone chahiye (`wheat_yellow_rust`, `cotton_leaf_curl`, `healthy_wheat` …).
Public datasets (PlantCity, cotton, rice) ki photos bhi inhi naamon ke folders mein daal kar zip mein shamil kar sakte hain.

In [ ]:
!pip -q install tensorflowjs==4.22.0
import tensorflow as tf, os, json, zipfile, shutil
print(tf.__version__, tf.config.list_physical_devices('GPU'))

## 1. dataset.zip upload karein

In [ ]:
from google.colab import files
up = files.upload()            # dataset.zip chunein
name = list(up.keys())[0]
shutil.rmtree('dataset', ignore_errors=True)
with zipfile.ZipFile(name) as z: z.extractall('.')
# zip ke andar 'dataset/' folder ho ya seedhe label folders — dono chalenge
root = 'dataset' if os.path.isdir('dataset') else '.'
classes = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)) and not d.startswith('.') and d != '__MACOSX')
for c in classes: print(c, len(os.listdir(os.path.join(root, c))))

## 2. Data tayyar (80% training, 20% check)

In [ ]:
IMG = 224
BATCH = 32
train = tf.keras.utils.image_dataset_from_directory(root, validation_split=0.2, subset='training', seed=42,
          image_size=(IMG, IMG), batch_size=BATCH, class_names=classes)
val = tf.keras.utils.image_dataset_from_directory(root, validation_split=0.2, subset='validation', seed=42,
          image_size=(IMG, IMG), batch_size=BATCH, class_names=classes)

aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.2),
])
prep = tf.keras.applications.mobilenet_v2.preprocess_input   # app bhi yahi karti hai (-1..1)
AUTO = tf.data.AUTOTUNE
train_ds = train.map(lambda x, y: (prep(aug(x, training=True)), y), num_parallel_calls=AUTO).prefetch(AUTO)
val_ds = val.map(lambda x, y: (prep(x), y), num_parallel_calls=AUTO).prefetch(AUTO)

## 3. MobileNetV2 + transfer learning

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False
inp = tf.keras.Input((IMG, IMG, 3))
x = base(inp, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(len(classes), activation='softmax')(x)
model = tf.keras.Model(inp, out)
model.compile(tf.keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=10)

# Fine-tune: upar ki layers bhi thodi sikhayein
base.trainable = True
for l in base.layers[:-40]: l.trainable = False
model.compile(tf.keras.optimizers.Adam(1e-5), 'sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=8,
          callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

## 4. Kitna sahi hai? (har bimari ka alag)

In [ ]:
import numpy as np
y_true, y_pred = [], []
for x, y in val_ds:
    y_true += list(y.numpy()); y_pred += list(model.predict(x, verbose=0).argmax(1))
y_true, y_pred = np.array(y_true), np.array(y_pred)
print('Overall accuracy:', (y_true == y_pred).mean().round(3))
for i, c in enumerate(classes):
    m = y_true == i
    if m.sum(): print(f'{c:28s} {((y_pred[m] == i).mean()):.2f}  ({m.sum()} photos)')

## 5. App ke liye export (TensorFlow.js)

In [ ]:
shutil.rmtree('saved', ignore_errors=True); shutil.rmtree('web_model', ignore_errors=True)
model.export('saved')
!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model --quantize_uint8 saved web_model
meta = {'format': 'graph', 'size': IMG, 'labels': classes, 'normalize': 'mobilenet_v2',
        'trainedAt': __import__('datetime').datetime.utcnow().isoformat() + 'Z'}
json.dump(meta, open('web_model/meta.json', 'w'), indent=2)
shutil.make_archive('kisan_model', 'zip', 'web_model')
files.download('kisan_model.zip')
print('kisan_model.zip → unzip → files ko app ke public/model/ mein daalein → npm run build')